<a href="https://colab.research.google.com/github/Malik-Zain-555/fake_news_style_detector/blob/main/fake_news_style_detector.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import kagglehub
import shutil
import os

# Download latest version
path = kagglehub.dataset_download("clmentbisaillon/fake-and-real-news-dataset")

# Copy the CSV files to the current working directory
for file in os.listdir(path):
    if file.endswith('.csv'):
        shutil.copy(os.path.join(path, file), './' + file)
        print(f"Saved: {file} to current working directory.")

Using Colab cache for faster access to the 'fake-and-real-news-dataset' dataset.
Saved: True.csv to current working directory.
Saved: Fake.csv to current working directory.


In [ ]:
import pandas as pd

df_fake = pd.read_csv("Fake.csv")
df_true = pd.read_csv("True.csv")

print(df_fake.shape)
print(df_true.shape)

(23481, 4)
(21417, 4)


In [ ]:
df_fake.head()

,title,text,subject,date
0,Donald Trump Sends Out Embarrassing New Year’...,Donald Trump just couldn t wish all Americans ...,News,"December 31, 2017"
1,Drunk Bragging Trump Staffer Started Russian ...,House Intelligence Committee Chairman Devin Nu...,News,"December 31, 2017"
2,Sheriff David Clarke Becomes An Internet Joke...,"On Friday, it was revealed that former Milwauk...",News,"December 30, 2017"
3,Trump Is So Obsessed He Even Has Obama’s Name...,"On Christmas day, Donald Trump announced that ...",News,"December 29, 2017"
4,Pope Francis Just Called Out Donald Trump Dur...,Pope Francis used his annual Christmas Day mes...,News,"December 25, 2017"


In [ ]:
df_true.head()

,title,text,subject,date
0,"As U.S. budget fight looms, Republicans flip t...",WASHINGTON (Reuters) - The head of a conservat...,politicsNews,"December 31, 2017"
1,U.S. military to accept transgender recruits o...,WASHINGTON (Reuters) - Transgender people will...,politicsNews,"December 29, 2017"
2,Senior U.S. Republican senator: 'Let Mr. Muell...,WASHINGTON (Reuters) - The special counsel inv...,politicsNews,"December 31, 2017"
3,FBI Russia probe helped by Australian diplomat...,WASHINGTON (Reuters) - Trump campaign adviser ...,politicsNews,"December 30, 2017"
4,Trump wants Postal Service to charge 'much mor...,SEATTLE/WASHINGTON (Reuters) - President Donal...,politicsNews,"December 29, 2017"


In [ ]:
df_fake['fake_news'] = 'Yes'
df_true['fake_news'] = 'No'

In [ ]:
combined_df = pd.concat([df_true, df_fake])
combined_df.head()

,title,text,subject,date,fake_news
0,"As U.S. budget fight looms, Republicans flip t...",WASHINGTON (Reuters) - The head of a conservat...,politicsNews,"December 31, 2017",No
1,U.S. military to accept transgender recruits o...,WASHINGTON (Reuters) - Transgender people will...,politicsNews,"December 29, 2017",No
2,Senior U.S. Republican senator: 'Let Mr. Muell...,WASHINGTON (Reuters) - The special counsel inv...,politicsNews,"December 31, 2017",No
3,FBI Russia probe helped by Australian diplomat...,WASHINGTON (Reuters) - Trump campaign adviser ...,politicsNews,"December 30, 2017",No
4,Trump wants Postal Service to charge 'much mor...,SEATTLE/WASHINGTON (Reuters) - President Donal...,politicsNews,"December 29, 2017",No


In [ ]:
from sklearn.utils import shuffle

df = shuffle(combined_df).reset_index(drop="true")

df.head()

,title,text,subject,date,fake_news
0,"LEFTIST Resentment Peddler, Larry David (Net W...",And why wouldn t he? The leftist comedy writer...,left-news,"Nov 8, 2015",Yes
1,Austria's prospective ruling parties agree on ...,VIENNA (Reuters) - Austria s conservative Peop...,worldnews,"November 3, 2017",No
2,Nazi Website Praises Trump’s Pathetic Respons...,This is proof that Donald Trump has only embol...,News,"August 13, 2017",Yes
3,"Fed chair choice down to Powell, Taylor, one s...",WASHINGTON (Reuters) - President Donald Trump’...,politicsNews,"October 26, 2017",No
4,Zimbabwe's Mugabe in talks with ousted vice pr...,JOHANNESBURG (Reuters) - Zimbabwe s top genera...,worldnews,"November 20, 2017",No


In [ ]:
print(df.columns)
print(df.shape)

Index(['title', 'text', 'subject', 'date', 'fake_news'], dtype='object')
(44898, 5)


In [ ]:
import re

def text_cleaner(txt):
  txt = txt.lower()

  txt = re.sub(r'http\S+ | www\.\S+'," ", txt)

  txt = re.sub(r'\S+@\S+', " ", txt)

  txt = re.sub(r'[^a-zA-Z\s]', " ", txt)

  txt = re.sub(r'reuters', " ", txt)

  txt = re.sub(r'\s', " ", txt).strip()

  return txt

In [ ]:
df['text'] = df['text'].apply(text_cleaner)
df.head(1)

,title,text,subject,date,fake_news
0,"LEFTIST Resentment Peddler, Larry David (Net W...",and why wouldn t he the leftist comedy writer...,left-news,"Nov 8, 2015",Yes


In [ ]:
x = df['text']
y = df['fake_news']

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer

x_train, x_test, y_train, y_test = train_test_split(
    x, y, test_size=0.2, random_state=42, stratify=y
)

In [ ]:
x_train.shape, x_test.shape

((35918,), (8980,))

In [ ]:
tfidf = TfidfVectorizer()

tfidf_x_train = tfidf.fit_transform(x_train)
tfidf_x_test = tfidf.transform(x_test)

In [ ]:
print(tfidf_x_train.shape)
print(tfidf_x_test.shape)

(35918, 97671)
(8980, 97671)


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score
logReg = LogisticRegression(max_iter=1000, random_state=42)

logReg.fit(tfidf_x_train,y_train)

y_pred = logReg.predict(tfidf_x_test)

print("Accuraccy Score: ", accuracy_score(y_test, y_pred))
print("Classification_Report: ")
print(classification_report(y_test, y_pred))

Accuraccy Score:  0.9820712694877506
Classification_Report: 
              precision    recall  f1-score   support

          No       0.98      0.98      0.98      4284
         Yes       0.98      0.98      0.98      4696

    accuracy                           0.98      8980
   macro avg       0.98      0.98      0.98      8980
weighted avg       0.98      0.98      0.98      8980



In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.model_selection import cross_val_score

pip = Pipeline([
    ('tfidf', TfidfVectorizer(stop_words="english", ngram_range=(1,2), max_features=3000)),
    ('logReg', LogisticRegression(max_iter=1000, random_state=42))
])

score = cross_val_score(pip, x_train, y_train, cv=5, n_jobs=1)
print(score.mean().round(4), score.std().round(4))

0.9786 0.0006


In [ ]:
s_f = df_fake['text'].str.contains("Reuters")
print(df_fake.groupby(s_f)['fake_news'].size())

s_t = df_true['text'].str.contains("Reuters")
print(df_true.groupby(s_t)['fake_news'].size())

text
False    23170
True       311
Name: fake_news, dtype: int64
text
False       39
True     21378
Name: fake_news, dtype: int64


In [ ]:
logReg.coef_.shape

(1, 97671)

In [ ]:
import numpy as np

print("Classes:", logReg.classes_)

# For binary classification, logReg.coef_ has shape (1, n_features)
coefs = logReg.coef_[0]

# Sort the coefficients
sorted_indices = np.argsort(coefs)

# Highly negative coefficients point to 'No' (True News)
true_news = sorted_indices[:10]

# Highly positive coefficients point to 'Yes' (Fake News)
false_news = sorted_indices[-10:][::-1]

Classes: ['No' 'Yes']


In [ ]:
print(sorted_indices)
print(tfidf.get_feature_names_out()[true_news])
print(tfidf.get_feature_names_out()[false_news])

[74301 60686 93142 ... 85603 90675 91747]
['said' 'on' 'washington' 'in' 'wednesday' 'republican' 'thursday'
 'tuesday' 'told' 'minister']
['via' 'us' 'this' 'read' 'mr' 'just' 'that' 'gop' 'com' 'obama']


Testing on custom news

In [ ]:
custom_df = ["WASHINGTON - The International Space Station orbits the Earth about once every 90 minutes, NASA officials said on Wednesday, adding that the crew sees around 16 sunrises and sunsets each day. An agency spokesman told reporters that the station has been continuously occupied since 2000.", "READ: You won't BELIEVE this! The International Space Station goes around the Earth every 90 minutes and the crew sees 16 sunrises a day. Just in, via NASA. Image: NASA. Click here to read more and share with your friends.", "WASHINGTON - Astronomers said on Wednesday the Moon has moved about 10,000 km closer to the Earth this year, a shift that officials told reporters is causing stronger tides worldwide. A government spokesman said further measurements would be released on Thursday."]

cdf = [text_cleaner(txt) for txt in custom_df]

Pdf = Pipeline([
    ('tfidf', TfidfVectorizer(stop_words='english', max_features=3000, ngram_range=(1,2))),
    ('logReg', LogisticRegression(random_state=42, max_iter=1000))
])

Pdf.fit(x_train, y_train)
print(Pdf.predict(cdfz))

score = cross_val_score(Pdf, x_train, y_train, cv=5, n_jobs=1)
print(score.mean(), score.std())

['No' 'Yes' 'No']
0.9785623000297349 0.000602884500664984


## What did the model learn?

- Accuracy: 98.8% (98.2% after removing "reuters"). It looks great, but it's misleading.
- The model learned the writing style of the source, not whether news is true.
- "Reuters" appears in 99.8% of real articles (21,378 of 21,417) and in only 311 fake ones.
- Removing "reuters" dropped accuracy by only ~0.6 points, so other style words carry the same signal.
- Top words are style words: real = said, washington, wednesday, told, minister. Fake = via, read, com, just.
- My own test: 2 of 3 texts were wrong (true blog-style news -> fake, false agency-style news -> real).
- Fix ideas (not done yet): use data from many sources, remove style markers like datelines, test on outside articles.